# 03. Biểu diễn BERT (B0–B5) và A3 (TV3)

Tính 6 embedding, lưu vào `embeddings/`, chạy A3 (BERT + K-Means, k = 6), ghi `results.csv` và cột `bert_kmeans` của `doc_topics.csv`.

Notebook chỉ gọi hàm trong `src/`. Embedding đã có file thì **không tính lại**.

In [28]:
import sys; sys.path.append("..")
from src.config import *

In [29]:
import time
import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocess import load_articles, get_tokens
from src.embed import compute_all, load_embedding, embed_query, run_bert_kmeans
from src.baselines import top_words_by_cluster
from src.metrics import evaluate, make_row, save_results

## 1. Dữ liệu

In [30]:
df = load_articles()
print(df.shape)
print(df["true_label"].value_counts().to_string())

(1800, 11)
true_label
thoi_su      300
suc_khoe     300
kinh_te      300
cong_nghe    300
the_thao     300
giai_tri     300


## 2. Tính embedding B0–B5

Mỗi khóa tính một lần và lưu `embeddings/<khóa>.npy`; khóa đã có file sẽ được bỏ qua. Muốn tính lại một khóa: `compute_all(df, keys=["b3_minilm"], overwrite=True)`.

Lần đầu mỗi mô hình sẽ tải từ Hugging Face (cần mạng). Thời gian tải **không** được tính vào thời gian embedding.

In [31]:
time_table = compute_all(df)
display(time_table)

[b0_tfidf_svd] đã có b0_tfidf_svd.npy, bỏ qua (overwrite=True để tính lại)
[b1_mbert_cls] đã có b1_mbert_cls.npy, bỏ qua (overwrite=True để tính lại)
[b2_mbert_mean] đã có b2_mbert_mean.npy, bỏ qua (overwrite=True để tính lại)
[b3_minilm] 1800 văn bản, cột text_bert


C:\Users\sincr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sincr\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.wa

[b3_minilm] xong: (1800, 384), 37.4s
[b4_vi_sbert] 1800 văn bản, cột text_bert_seg


C:\Users\sincr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sincr\.cache\huggingface\hub\models--keepitreal--vietnamese-sbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████

[b4_vi_sbert] xong: (1800, 768), 104.4s
[b5_e5] 1800 văn bản, cột text_bert


C:\Users\sincr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sincr\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|████

[b5_e5] xong: (1800, 768), 129.1s


,embedding,n_docs,dim,time_s
0,b0_tfidf_svd,1800,100,1.1355
1,b1_mbert_cls,1800,768,182.6839
2,b2_mbert_mean,1800,768,173.1924
3,b3_minilm,1800,384,37.4062
4,b4_vi_sbert,1800,768,104.4018
5,b5_e5,1800,768,129.1212


## 3. Kiểm tra embedding

In [32]:
for key in EMBEDDINGS:
    X = load_embedding(key)                      # tự kiểm tra số dòng bằng số bài
    norms = np.linalg.norm(X, axis=1)
    print(f"{key:15s} shape={X.shape} dtype={X.dtype} NaN={np.isnan(X).any()} "
          f"norm trong [{norms.min():.3f}, {norms.max():.3f}]")
    assert X.shape[0] == len(df) and not np.isnan(X).any() and np.allclose(norms, 1, atol=1e-3)
print("Các file embedding hợp lệ.")

b0_tfidf_svd    shape=(1800, 100) dtype=float32 NaN=False norm trong [1.000, 1.000]
b1_mbert_cls    shape=(1800, 768) dtype=float32 NaN=False norm trong [1.000, 1.000]
b2_mbert_mean   shape=(1800, 768) dtype=float32 NaN=False norm trong [1.000, 1.000]
b3_minilm       shape=(1800, 384) dtype=float32 NaN=False norm trong [1.000, 1.000]
b4_vi_sbert     shape=(1800, 768) dtype=float32 NaN=False norm trong [1.000, 1.000]
b5_e5           shape=(1800, 768) dtype=float32 NaN=False norm trong [1.000, 1.000]
Các file embedding hợp lệ.


Kiểm tra `embed_query` (app demo sẽ dùng): vector của một bài tính lại từ văn bản phải gần như trùng vector đã lưu (cosine ≈ 1).

In [33]:
i = 0
for key in EMBEDDINGS:
    if EMBEDDINGS[key]["model"] is None:
        continue                                 # B0 không nhúng được văn bản mới
    q = embed_query(df.loc[i, "text_bert"], key)[0]
    cos = float(q @ load_embedding(key)[i])
    print(f"{key:15s} cosine(embed_query, đã lưu) = {cos:.4f}", "" if cos > 0.99 else "  <-- LỆCH, kiểm tra embed_query")

    1/1
b1_mbert_cls    cosine(embed_query, đã lưu) = 1.0000 
    1/1
b2_mbert_mean   cosine(embed_query, đã lưu) = 1.0000 


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.02it/s]


b3_minilm       cosine(embed_query, đã lưu) = 1.0000 


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.22it/s]


b4_vi_sbert     cosine(embed_query, đã lưu) = 1.0000 


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.31it/s]

b5_e5           cosine(embed_query, đã lưu) = 1.0000 


## 4. A3: BERT + K-Means (k = 6)

`time_s` = thời gian embedding + thời gian K-Means (để so sánh được với LSA, vốn gồm cả bước biểu diễn).

In [34]:
emb_time = pd.read_csv(EMB_TIME_CSV).set_index("embedding")["time_s"]
tokens = get_tokens(df)
text_tok = df["text_tok"].tolist()
y_true = df["true_label"].values


def chay_a3(key, seed):
    X = load_embedding(key)
    t0 = time.perf_counter()
    labels = run_bert_kmeans(X, k=N_TOPICS, seed=seed)
    t_km = time.perf_counter() - t0
    words = top_words_by_cluster(text_tok, labels)
    scores = evaluate(labels, y_true, words, tokens)
    row = make_row("bert_kmeans", key, "k6", "", seed, scores, float(emb_time[key]) + t_km)
    return labels, words, row

In [35]:
rows = [chay_a3(key, SEED)[2] for key in EMBEDDINGS]
save_results(rows)
df_a3 = (pd.DataFrame(rows)[["embedding", "nmi", "ari", "purity", "cv", "diversity", "time_s"]]
         .sort_values("nmi", ascending=False).reset_index(drop=True))
display(df_a3)

,embedding,nmi,ari,purity,cv,diversity,time_s
0,b0_tfidf_svd,0.6646,0.5425,0.7406,0.7920,0.9500,1.4161
1,b5_e5,0.5707,0.5226,0.7200,0.6864,0.9667,129.4701
2,b3_minilm,0.4531,0.3848,0.6306,0.7709,0.8667,37.5942
3,b2_mbert_mean,0.4389,0.3742,0.6350,0.7114,0.9500,173.5187
4,b4_vi_sbert,0.4179,0.3149,0.5706,0.7173,0.9500,104.7828
5,b1_mbert_cls,0.1652,0.1337,0.3467,0.4450,0.8500,183.1013


## 5. Chọn embedding BERT tốt nhất (NMI cao nhất, k = 6) và chạy 3 seed

B0 không dùng BERT nên không tham gia chọn cho A3.

In [36]:
bert_keys = [k for k in EMBEDDINGS if k != "b0_tfidf_svd"]
best = df_a3[df_a3["embedding"].isin(bert_keys)].iloc[0]["embedding"]
print("Embedding BERT tốt nhất (NMI, k=6):", best, "| BEST_EMBEDDING hiện tại trong config:", BEST_EMBEDDING)

rows_seed, labels_best = [], None
for s in SEEDS:
    labels, words, row = chay_a3(best, s)
    rows_seed.append(row)
    if s == SEED:
        labels_best, words_best = labels, words
save_results(rows_seed)

df_seed = pd.DataFrame(rows_seed)
summary = df_seed[["ari", "nmi", "purity", "cv", "diversity"]].agg(["mean", "std"]).round(4)
display(summary)

Embedding BERT tốt nhất (NMI, k=6): b5_e5 | BEST_EMBEDDING hiện tại trong config: b3_minilm


,ari,nmi,purity,cv,diversity
mean,0.5312,0.5838,0.7239,0.7196,0.9611
std,0.0124,0.0197,0.0082,0.0434,0.0096


## 6. Ghi nhãn vào `doc_topics.csv`

Chỉ thêm/cập nhật cột `bert_kmeans` (nhãn của cấu hình tốt nhất, seed 42). Nếu file đã có thì giữ nguyên các cột của người khác.

In [37]:
labels_str = pd.Series(labels_best).astype(int).astype(str).values
if DOC_TOPICS_CSV.exists():
    dt = pd.read_csv(DOC_TOPICS_CSV, dtype=str, keep_default_na=False)   # giữ nguyên chữ, không đổi kiểu cột người khác
    assert (dt["id"].values == df["id"].values).all(), "doc_topics.csv khác thứ tự với articles.csv"
else:
    dt = pd.DataFrame({"id": df["id"], "true_label": df["true_label"]})
dt["bert_kmeans"] = labels_str
TABLES_DIR.mkdir(parents=True, exist_ok=True)
dt.to_csv(DOC_TOPICS_CSV, index=False, encoding="utf-8")
print("Đã ghi cột bert_kmeans vào", DOC_TOPICS_CSV.name, "| các cột:", list(dt.columns))

Đã ghi cột bert_kmeans vào doc_topics.csv | các cột: ['id', 'true_label', 'lda', 'lsa_kmeans', 'bert_kmeans']


## 7. Xem chủ đề tìm được (cấu hình tốt nhất)

In [38]:
for t, w in enumerate(words_best):
    print(f"Chủ đề {t}: {' '.join(w)}")
display(pd.crosstab(pd.Series(labels_best, name="cụm"), pd.Series(y_true, name="nhãn thật")))

Chủ đề 0: doanh_nghiệp việt_nam công_nghệ đầu_tư kinh_tế dự_án nước tỷ xây_dựng sản_xuất
Chủ đề 1: nvidia huang iphone dùng công_ty ceo thiết_bị apple công_nghệ màn_hình
Chủ đề 2: đồng giá giảm triệu usd vàng xã đất tăng công_an
Chủ đề 3: trận đội đội_tuyển bóng cầu_thủ hlv việt_nam u17 giải thắng
Chủ đề 4: bác_sĩ người_bệnh bệnh điều_trị ung_thư thuốc thận đau bệnh_viện ăn
Chủ đề 5: diễn_viên phim ca_sĩ nghệ_sĩ khán_giả đẹp vai nữ chương_trình văn_hóa


nhãn thật,cong_nghe,giai_tri,kinh_te,suc_khoe,the_thao,thoi_su
cụm,,,,,,
0,49,25,136,20,17,116
1,226,11,57,0,1,1
2,17,2,102,16,3,157
3,3,2,0,0,265,1
4,2,5,1,257,6,13
5,3,255,4,7,8,12


## 8. Bốn phép so sánh trọng tâm (điền nhận xét vào `ket_qua/03_bieu_dien_bert.md`)

In [39]:
nmi = df_a3.set_index("embedding")["nmi"]
print(f"B1 [CLS] {nmi['b1_mbert_cls']:.4f}  vs  B2 mean {nmi['b2_mbert_mean']:.4f}")
print(f"mBERT nguyên bản (tốt nhất B1/B2) {max(nmi['b1_mbert_cls'], nmi['b2_mbert_mean']):.4f}"
      f"  vs  SBERT (tốt nhất B3–B5) {nmi[['b3_minilm', 'b4_vi_sbert', 'b5_e5']].max():.4f}")
print(f"B3 đa ngôn ngữ {nmi['b3_minilm']:.4f}  vs  B4 tiếng Việt {nmi['b4_vi_sbert']:.4f}")
print(f"B0 tần suất {nmi['b0_tfidf_svd']:.4f}  vs  BERT tốt nhất {nmi[bert_keys].max():.4f}")
print("\nBáo TV4 và TV5 cập nhật BEST_EMBEDDING trong config thành:", best)

B1 [CLS] 0.1652  vs  B2 mean 0.4389
mBERT nguyên bản (tốt nhất B1/B2) 0.4389  vs  SBERT (tốt nhất B3–B5) 0.5707
B3 đa ngôn ngữ 0.4531  vs  B4 tiếng Việt 0.4179
B0 tần suất 0.6646  vs  BERT tốt nhất 0.5707

Báo TV4 và TV5 cập nhật BEST_EMBEDDING trong config thành: b5_e5
